# Single spin Part 3: Single qubit gate optimization using GRAPE

## 1. Generate a PWC pulse shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import GaussEnvelope
from paraqeet.signal.pwc_generator import PWCGenerator

First, let's generate a piecewise constant (PWC) pulse envelope for the Gaussian pulse

In [ ]:
t_final = 20e-9
tlist = np.linspace(0, t_final, 101)
tone = GaussEnvelope(amplitude=Quantity(2 * np.pi / t_final / 3, -5 * np.pi / t_final, 5 * np.pi / t_final))
tone.t_final.set_value(t_final)
gen = PWCGenerator(envelopes=[tone], tlist=tlist)
gen.multiply_flat_top = True

In [ ]:
params = gen.get_parameters()

In [ ]:
def plot_pulses():
    """Plots the pulses"""
    ts = np.linspace(0, t_final, 1001)

    _, ax = plt.subplots(1, figsize=(5, 3), sharex=True)

    ax.plot(ts / 1e-9, tone.get_value(ts) / 1e6, label="Smooth curve")
    ax.plot(ts / 1e-9, np.real(gen.get_value(ts)) / 1e6, ls="--", label="in-phase")
    ax.plot(
        ts / 1e-9,
        np.imag(gen.get_value(ts)) / 1e6,
        ls="--",
        label="out-of-phase",
    )

    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("Amplitude (MHz)")
    ax.legend()
    ax.grid()
    plt.show()


plot_pulses()

## 2. Define Hamiltonian in the rotating frame of drive

Next, we setup the qubit system we want to control. We define the Hamiltonian in the rotating frame of drive such that the pulse oscillates slowly to apply GRAPE gradients.

The Hamiltonain in the rotating frame of the drive is given by - 
$$ H(t) = \big(\omega_q - \omega_d\big) b^\dagger b -\frac{\alpha}{2} (b^\dagger)^2 b^2 + (\epsilon(t) b + \epsilon(t)^* b)$$

In [ ]:
from paraqeet.quantity import Quantity
from paraqeet.model.closed_system import ClosedSystem
from paraqeet.model.rotating_frame_drive import RotatingFrameDrive
from paraqeet.model.transmon import Transmon


freq = 7.86e9 * 2 * np.pi
dims = 3
anharm = -50e6 * 2 * np.pi
offset = 5e6 * 2 * np.pi

drive_freq = freq + offset
qubit_freq = freq - drive_freq

Drive = RotatingFrameDrive(gen)
transmon = Transmon(
    frequency=Quantity(
        qubit_freq,
        1.2 * qubit_freq,
        0.8 * qubit_freq,
        unit="Hz",
        name="Frequency",
    ),
    anharmonicity=Quantity(anharm, 1.2 * anharm, 0.8 * anharm, unit="Hz", name="Anharmonicity"),
    drives=[Drive],
    dimension=dims,
)

model = ClosedSystem(transmon)

In [ ]:
from paraqeet.measurement.state_transfer_fidelity import StateTransferFidelityGRAPE
from paraqeet.propagation.scipy_expm_grape import ScipyExpmGRAPE


prop = ScipyExpmGRAPE(model, res=1e9)

init = np.array([[1.0], [0.0], [0.0]])  # |0>
target = np.array([[0.0], [1.0], [0.0]])  # |1>

times = np.array([0.0, t_final])

prop.set_initial_state(init)
prop.target_state = target

prop.use_schirmer_derivative = True

zeroone = StateTransferFidelityGRAPE(
    propagation=prop,
    initial_state=init,
    target_state=target,
)

In [ ]:
def plot_states():
    """Plot the states."""
    ts = np.linspace(0, t_final, 1001)
    states = prop.propagate(ts)
    sig = gen.get_value(ts)

    _, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, np.real(sig), label="I")
    ax[0].plot(ts / 1e-9, np.imag(sig), label="Q")
    ax[0].legend(loc=1)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, np.abs(states)[:, :, 0] ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time (ns)")
    ax[0].grid()
    ax[1].grid()
    plt.show()


plot_states()

As expected, we get a partial transfer and a low fidelity.

In [ ]:
zeroone.measure(times)

## 3. Opimisation
We define an optimizer and link our fidelity measure as a goal function and the parameters of the cosine tone and optimize just amplitude and frequency, as in the state transfer example.

In [ ]:
from paraqeet.optimization_map import OptimizationMap
from paraqeet.optimizers.scipy_optimizer_gradient import ScipyOptimizerGradient

optmap = OptimizationMap()
optmap.add(gen, params)
opt = ScipyOptimizerGradient(zeroone, optimizables=optmap)

In [ ]:
opt.optimize(tlist)

In [ ]:
plot_pulses()

In [ ]:
plot_states()